<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">اصلاح کوچک روی وزن ثابت</h1>
<p style="text-align:right">درس 73 از 92 · برای تنظیم رفتار، لازم است همهٔ وزن‌ها تغییر کنند؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">65b-lora</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-02/65b-lora.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">شاخهٔ <bdi dir="ltr">LoRA</bdi> را بنویسید و ببینید کدام <bdi dir="ltr">Parameter</bdi> در گام اول <bdi dir="ltr">Gradient</bdi> می‌گیرد.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: ضرب</span> ماتریسی، <bdi dir="ltr">Linear</bdi>، <bdi dir="ltr">Tensor</bdi> ثابت و <bdi dir="ltr">Parameter</bdi> قابل آموزش.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۴۵–۸۰ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر <bdi dir="ltr">A</bdi> و <bdi dir="ltr">B</bdi> هر دو صفر باشند، آیا این شاخه از صفر حرکت می‌کند؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import torch
from torch import nn
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(12)
project_model = MiniGPT(ModelConfig(8,4,12,3,1,0.0))
W = project_model.language_model_head.weight.detach().clone()
A = nn.Parameter(torch.randn(2,12)*0.01)
B = nn.Parameter(torch.zeros(8,2))
x = torch.randn(3,12)
print('base weight shape:',tuple(W.shape),'adapter scalars:',A.numel()+B.numel())
print('base output shape:',tuple((x@W.T).shape))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">lora_forward(x,W,A,B,alpha)</code> خروجی <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">xWᵀ + (alpha/r)(xAᵀ)Bᵀ</code> را بدهد؛ <bdi dir="ltr">r</bdi> تعداد سطرهای <bdi dir="ltr">A</bdi> است. <bdi dir="ltr">W</bdi> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(M,C)</code>، <bdi dir="ltr">A</bdi> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(r,C)</code>، <bdi dir="ltr">B</bdi> شکل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(M,r)</code> دارد. هیچ <bdi dir="ltr">Tensor</bdi> را درجا تغییر ندهید.</p>
</div>

In [ ]:
def lora_forward(x, W, A, B, alpha):
    # TODO: شاخهٔ کم‌رتبه به خروجی پایه اضافه می‌شود
    return None

In [ ]:
def test_exercise():
    result = lora_forward(x,W,A,B,2.0)
    if result is None:
        return False
    torch.testing.assert_close(result,x@W.T)
    trial_b = torch.ones_like(B)
    torch.testing.assert_close(lora_forward(x,W,A,trial_b,4.0),x@(W+2.0*(trial_b@A)).T)
    target = torch.ones_like(result)
    ((result-target)**2).mean().backward()
    assert W.grad is None
    assert A.grad is not None and torch.count_nonzero(A.grad)==0
    assert B.grad is not None and B.grad.norm()>0
    assert A.numel()+B.numel()==40
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: lora_forward')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط <bdi dir="ltr">Rank</bdi> را عوض کنید و <bdi dir="ltr">Parameter</bdi>های شاخه را بشمارید. کاهش <bdi dir="ltr">Rank</bdi> به‌تنهایی کیفیت <bdi dir="ltr">Fine-Tuning</bdi> را پیش‌بینی نمی‌کند.</p>
</div>

In [ ]:
for rank in (1,2,4):
    a = torch.zeros(rank,12)
    b = torch.zeros(8,rank)
    print(rank,'adapter:',a.numel()+b.numel(),'base still needed:',W.numel())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">صفرکردن هر دو عامل، <bdi dir="ltr">Gradient</bdi> هرکدام را در عامل صفر دیگر ضرب می‌کند. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">initialize_adapter(out_features,in_features,rank)</code> دو <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">nn.Parameter</code> بسازد: <bdi dir="ltr">A</bdi> با مقدار تصادفی کوچک و <bdi dir="ltr">B</bdi> صفر.</p>
</div>

In [ ]:
wrong_a = nn.Parameter(torch.zeros(2,12))
wrong_b = nn.Parameter(torch.zeros(8,2))
((x@W.T+(x@wrong_a.T)@wrong_b.T-1)**2).mean().backward()
print('both-zero gradient norms:',wrong_a.grad.norm().item(),wrong_b.grad.norm().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def initialize_adapter(out_features, in_features, rank):
    # TODO: خروجی آغازین پایه بماند، اما شاخه بتواند یاد بگیرد
    return None

In [ ]:
def test_repair():
    result = initialize_adapter(8,12,2)
    if result is None:
        return False
    a,b = result
    assert isinstance(a,nn.Parameter) and isinstance(b,nn.Parameter)
    assert a.shape==(2,12) and b.shape==(8,2)
    assert torch.count_nonzero(a)>0 and torch.count_nonzero(b)==0
    a2,b2 = initialize_adapter(3,5,1)
    assert a2.shape==(1,5) and b2.shape==(3,1)
    assert a2.requires_grad and b2.requires_grad
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: initialize_adapter')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><bdi dir="ltr">W</bdi> یک کپی ثابت از <bdi dir="ltr">Language-model head</bdi> <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> است؛ این شاخه به ساختار خود مدل اضافه نشده است. <bdi dir="ltr">LoRA</bdi> روش انتخاب <bdi dir="ltr">Parameter</bdi>های قابل آموزش است و می‌تواند کنار هدف <bdi dir="ltr">SFT</bdi> به کار رود؛ هدف آموزشی جداگانه‌ای نیست.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا ۴۰ <bdi dir="ltr">Parameter</bdi> قابل آموزش به معنی ۴۰ <bdi dir="ltr">Parameter</bdi> لازم برای اجرای مدل نیست؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-02/65b-lora.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/65b-lora.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>